In [0]:
gold_tables = [
    "order_summary",
    "daily_sales",
    "product_performance",
    "customer_performance"
]

for table in gold_tables:
    df = spark.table(f"ecommerce.gold.{table}")
    print(f"{table}: {df.count()} rows")

order_summary: 12 rows
daily_sales: 12 rows
product_performance: 9 rows
customer_performance: 11 rows


In [0]:
display(spark.sql("SHOW TABLES IN ecommerce.gold"))

database,tableName,isTemporary
gold,customer_performance,false
gold,daily_sales,false
gold,order_summary,false
gold,product_performance,false


In [0]:
display(spark.sql("""
    SELECT *
    FROM ecommerce.gold.customer_performance
    ORDER BY total_spent DESC
"""))

customer_id,customer_name,email,country,state,signup_date,total_orders,total_spent,average_order_value
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18,2,6897,3448.5
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01,1,3997,3997.0
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15,2,3299,1649.5
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15,1,2298,2298.0
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01,1,2199,2199.0
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12,1,1899,1899.0
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02,1,1599,1599.0
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05,1,1499,1499.0
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20,1,899,899.0
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10,1,699,699.0


In [0]:
from pyspark.sql.functions import sum as spark_sum, countDistinct, avg

orders_df = spark.table("ecommerce.silver.orders")
customers_df = spark.table("ecommerce.silver.customers")

customer_metrics_df = (
    orders_df
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        spark_sum("order_total").alias("total_spent"),
        avg("order_total").alias("average_order_value")
    )
)

customer_performance_df = (
    customers_df
    .join(customer_metrics_df, "customer_id", "left")
    .select(
        "customer_id",
        "customer_name",
        "email",
        "country",
        "state",
        "signup_date",
        "total_orders",
        "total_spent",
        "average_order_value"
    )
    .orderBy("customer_id")
)

display(customer_performance_df)

customer_id,customer_name,email,country,state,signup_date,total_orders,total_spent,average_order_value
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15,1,2298,2298.0
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18,2,6897,3448.5
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02,1,1599,1599.0
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10,1,699,699.0
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15,2,3299,1649.5
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01,1,2199,2199.0
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12,1,1899,1899.0
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20,1,899,899.0
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01,1,3997,3997.0
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05,1,1499,1499.0


In [0]:
(
    customer_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.gold.customer_performance")
)

In [0]:
display(spark.sql("""
    SELECT *
    FROM ecommerce.gold.product_performance
    ORDER BY total_revenue DESC
"""))

product_id,product_name,category,subcategory,unit_price,total_quantity_sold,total_revenue,total_orders,stock_quantity
P007,Running Shoes,Fashion,Footwear,3299.0,2,6598.0,2,50
P010,Bluetooth Speaker,Electronics,Audio,2199.0,3,6597.0,2,65
P002,Mechanical Keyboard,Electronics,Accessories,2499.0,1,2499.0,1,75
P004,Laptop Stand,Electronics,Office,1899.0,1,1899.0,1,60
P008,Cotton T-Shirt,Fashion,Clothing,899.0,2,1798.0,2,150
P006,Backpack,Fashion,Bags,1599.0,1,1599.0,1,90
P001,Wireless Mouse,Electronics,Accessories,799.0,2,1598.0,2,100
P003,USB-C Hub,Electronics,Accessories,1499.0,1,1499.0,1,120
P005,Water Bottle,Home & Living,Kitchen,699.0,2,1398.0,3,200


In [0]:
from pyspark.sql.functions import sum as spark_sum, countDistinct

items_df = spark.table("ecommerce.silver.order_items")
products_df = spark.table("ecommerce.silver.products")

product_performance_df = (
    items_df
    .groupBy("product_id")
    .agg(
        spark_sum("quantity").alias("total_quantity_sold"),
        spark_sum("line_amount").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
    .join(products_df, "product_id", "left")
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "unit_price",
        "total_quantity_sold",
        "total_revenue",
        "total_orders",
        "stock_quantity"
    )
    .orderBy("product_id")
)

display(product_performance_df)

product_id,product_name,category,subcategory,unit_price,total_quantity_sold,total_revenue,total_orders,stock_quantity
P001,Wireless Mouse,Electronics,Accessories,799.0,2,1598.0,2,100
P002,Mechanical Keyboard,Electronics,Accessories,2499.0,1,2499.0,1,75
P003,USB-C Hub,Electronics,Accessories,1499.0,1,1499.0,1,120
P004,Laptop Stand,Electronics,Office,1899.0,1,1899.0,1,60
P005,Water Bottle,Home & Living,Kitchen,699.0,2,1398.0,3,200
P006,Backpack,Fashion,Bags,1599.0,1,1599.0,1,90
P007,Running Shoes,Fashion,Footwear,3299.0,2,6598.0,2,50
P008,Cotton T-Shirt,Fashion,Clothing,899.0,2,1798.0,2,150
P010,Bluetooth Speaker,Electronics,Audio,2199.0,3,6597.0,2,65


In [0]:
(
    product_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.gold.product_performance")
)

In [0]:
display(spark.sql("""
    SELECT *
    FROM ecommerce.gold.daily_sales
    ORDER BY order_date
"""))

order_date,total_sales,total_orders
2026-09-01,2298,1
2026-09-02,2499,1
2026-09-03,1599,1
2026-09-04,699,1
2026-09-05,3299,1
2026-09-06,2199,1
2026-09-07,1899,1
2026-09-08,899,1
2026-09-09,3997,1
2026-09-10,1499,1


In [0]:
from pyspark.sql.functions import sum as spark_sum, countDistinct

orders_df = spark.table("ecommerce.silver.orders")

daily_sales_df = (
    orders_df
    .groupBy("order_date")
    .agg(
        spark_sum("order_total").alias("total_sales"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy("order_date")
)

display(daily_sales_df)

order_date,total_sales,total_orders
2026-09-01,2298,1
2026-09-02,2499,1
2026-09-03,1599,1
2026-09-04,699,1
2026-09-05,3299,1
2026-09-06,2199,1
2026-09-07,1899,1
2026-09-08,899,1
2026-09-09,3997,1
2026-09-10,1499,1


In [0]:
(
    daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.gold.daily_sales")
)

In [0]:
display(spark.sql("""
    SELECT *
    FROM ecommerce.gold.order_summary
    ORDER BY order_date
"""))

order_id,customer_id,order_date,order_status,order_total,total_payment_amount,payment_count
O1001,C001,2026-09-01,Delivered,2298,2298.0,1
O1002,C002,2026-09-02,Shipped,2499,2499.0,1
O1003,C003,2026-09-03,Delivered,1599,1599.0,1
O1004,C004,2026-09-04,Cancelled,699,699.0,1
O1005,C005,2026-09-05,Delivered,3299,3299.0,1
O1006,C006,2026-09-06,Processing,2199,2199.0,1
O1007,C007,2026-09-07,Delivered,1899,1899.0,1
O1008,C008,2026-09-08,Returned,899,899.0,1
O1009,C009,2026-09-09,Delivered,3997,3997.0,1
O1010,C010,2026-09-10,Shipped,1499,1499.0,1


In [0]:
from pyspark.sql.functions import sum as spark_sum, countDistinct

orders = spark.table("ecommerce.silver.orders").alias("o")

payments = (
    spark.table("ecommerce.silver.payments")
    .groupBy("order_id")
    .agg(
        spark_sum("amount").alias("total_payment_amount"),
        countDistinct("payment_id").alias("payment_count")
    )
    .alias("p")
)

order_summary_df = (
    orders.join(payments, "order_id", "left")
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
        "order_total",
        "total_payment_amount",
        "payment_count"
    )
)

display(order_summary_df)

order_id,customer_id,order_date,order_status,order_total,total_payment_amount,payment_count
O1001,C001,2026-09-01,Delivered,2298,2298.0,1
O1002,C002,2026-09-02,Shipped,2499,2499.0,1
O1003,C003,2026-09-03,Delivered,1599,1599.0,1
O1004,C004,2026-09-04,Cancelled,699,699.0,1
O1005,C005,2026-09-05,Delivered,3299,3299.0,1
O1006,C006,2026-09-06,Processing,2199,2199.0,1
O1007,C007,2026-09-07,Delivered,1899,1899.0,1
O1008,C008,2026-09-08,Returned,899,899.0,1
O1009,C009,2026-09-09,Delivered,3997,3997.0,1
O1010,C010,2026-09-10,Shipped,1499,1499.0,1


In [0]:
(
    order_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.gold.order_summary")
)

In [0]:
display(spark.sql("""
SELECT *
FROM ecommerce.gold.daily_sales
ORDER BY order_date
"""))

order_date,total_sales,total_orders
2026-09-01,2298,1
2026-09-02,2499,1
2026-09-03,1599,1
2026-09-04,699,1
2026-09-05,3299,1
2026-09-06,2199,1
2026-09-07,1899,1
2026-09-08,899,1
2026-09-09,3997,1
2026-09-10,1499,1


In [0]:
display(spark.sql("""
SELECT *
FROM ecommerce.gold.customer_performance
ORDER BY total_spent DESC
"""))

customer_id,customer_name,email,country,state,signup_date,total_orders,total_spent,average_order_value
C002,Rahul Menon,rahul.menon@example.com,India,Karnataka,2026-01-18,2,6897,3448.5
C009,Nisha George,nisha.george@example.com,India,Kerala,2026-04-01,1,3997,3997.0
C005,Diya Joseph,diya.joseph@example.com,India,Kerala,2026-02-15,2,3299,1649.5
C001,Ananya Nair,ananya.nair@example.com,India,Kerala,2026-01-15,1,2298,2298.0
C006,Vivek Shah,vivek.shah@example.com,India,Gujarat,2026-03-01,1,2199,2199.0
C007,Sneha Iyer,sneha.iyer@example.com,India,Karnataka,2026-03-12,1,1899,1899.0
C003,Meera Thomas,meera.thomas@example.com,India,Tamil Nadu,2026-02-02,1,1599,1599.0
C010,Aditya Rao,aditya.rao@example.com,India,Telangana,2026-04-05,1,1499,1499.0
C008,Kiran Das,kiran.das@example.com,India,West Bengal,2026-03-20,1,899,899.0
C004,Arjun Kumar,arjun.kumar@example.com,India,Maharashtra,2026-02-10,1,699,699.0


In [0]:
display(spark.sql("""
SELECT *
FROM ecommerce.gold.product_performance
ORDER BY total_revenue DESC
"""))

product_id,product_name,category,subcategory,unit_price,total_quantity_sold,total_revenue,total_orders,stock_quantity
P007,Running Shoes,Fashion,Footwear,3299.0,2,6598.0,2,50
P010,Bluetooth Speaker,Electronics,Audio,2199.0,3,6597.0,2,65
P002,Mechanical Keyboard,Electronics,Accessories,2499.0,1,2499.0,1,75
P004,Laptop Stand,Electronics,Office,1899.0,1,1899.0,1,60
P008,Cotton T-Shirt,Fashion,Clothing,899.0,2,1798.0,2,150
P006,Backpack,Fashion,Bags,1599.0,1,1599.0,1,90
P001,Wireless Mouse,Electronics,Accessories,799.0,2,1598.0,2,100
P003,USB-C Hub,Electronics,Accessories,1499.0,1,1499.0,1,120
P005,Water Bottle,Home & Living,Kitchen,699.0,2,1398.0,3,200


In [0]:
display(spark.sql("""
SELECT *
FROM ecommerce.gold.order_summary
LIMIT 5
"""))

order_id,customer_id,order_date,order_status,order_total,total_payment_amount,payment_count
O1001,C001,2026-09-01,Delivered,2298,2298.0,1
O1002,C002,2026-09-02,Shipped,2499,2499.0,1
O1003,C003,2026-09-03,Delivered,1599,1599.0,1
O1004,C004,2026-09-04,Cancelled,699,699.0,1
O1005,C005,2026-09-05,Delivered,3299,3299.0,1
